# 제어 흐름: 메타 루프 vs 스테이지드 루프

cutlass 커널은 호스트에서 한 번 **트레이스**(파이썬 본문이 실행되며 IR을 만듦)된 다음 GPU에서
여러 번 **실행**됩니다. 여러분이 쓰는 루프는 전부 이 경계선의 한쪽에 속합니다:

- **메타 루프**(`cutlass.range_constexpr`)는 *메타프로그램*입니다 — **파이썬 인터프리터가,
  호스트에서, 트레이싱 중에** 실행합니다. 본문이 그 자리에서 `N`번 걸어지고, IR `N`벌이 그대로
  나옵니다(완전 언롤). 진짜 파이썬이라서 본문은 *파이썬이 할 수 있는 건 뭐든* 할 수 있습니다 —
  `int`로 계산, 분기, 심지어 `raise`까지.
- **스테이지드 루프**(`cutlass.range`)는 본문을 **한 번만** 트레이스해서 GPU `scf.for` 하나로
  만들고, 이걸 **하드웨어**가 런타임에 반복합니다. 루프 인덱스는 파이썬 `int`가 아니라 *스테이지드*
  값입니다 — 아직 쥘 수 있는 구체적 숫자가 없습니다.

이 경계를 가장 깔끔하게 *보는* 방법은 트레이스타임 `print`입니다 — 호스트가 본문을 걸을 때마다
한 번씩 찍히니, 반복 횟수를 세어주는 셈입니다. 이번 장은 데이터나 배열이 필요 없습니다 — 루프와
print뿐입니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/1_dsl_features/02_control_flow.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/1_dsl_features/02_control_flow.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** 제어 흐름을 가르는 **트레이스타임 vs 런타임** 경계선; 메타 루프
**`cutlass.range_constexpr(n)`**(본문이 *파이썬에서* `n`번 실행, 완전 언롤, bound는 반드시
파이썬 `int`) vs 스테이지드 루프 **`cutlass.range(n)`**(본문을 GPU `scf.for`로 *한 번만*
트레이스, bound는 동적 `Int32`도 가능); 메타 루프가 왜 **파이썬을 마음대로** 쓸 수 있는지
(트레이싱 도중 `raise` 한 번 만들어봄); 부분 언롤 옵션 **`cutlass.range(n, unroll=K)`**;
컴파일타임 분기 **`if cutlass.const_expr(cond):`** vs 그냥 런타임 `if`; 그리고 이 모든 걸
`print()`(호스트, 트레이스타임) vs `cute.printf()`(디바이스, 런타임)만 보고 읽어내는 법.

**실행 환경:** 공식 노트북 원문 기준 CUDA GPU면 어느 것이든 동작합니다(단, `Arch` enum이
`sm_80`부터 시작해 T4는 해당 없음 — README 참고). 이 노트북은 **Colab L4**에서 실제로 실행해
검증했습니다.

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl==4.8.0`으로 고정합니다(노트북1과 동일 버전).

**함정(노트북1에서 이미 확인함): 같은 커널에서 설치 직후 import하면 실패합니다.** `.pth`
파일이 인터프리터 시작 시점에만 읽히기 때문 — 설치 셀 다음에 반드시 `site.main()` 셀을
넣습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

In [ ]:
import cutlass
import cutlass.cute as cute

cutlass.cuda.initialize_cuda_context()   # 프로세스당 한 번, launch 전에
print("imports OK")

### 함정: `cute.printf` 출력이 셀에 기본적으로 안 뜹니다 (노트북1 참고)

노트북1에서 확인한 대로, `cute.printf`(디바이스 출력) 결과가 이 Colab 환경에서는 셀 출력에
기본적으로 나타나지 않습니다(원인은 확인 못 함 — 노트북1의 실측 검증 절 참고). 이 챕터는
`cute.printf`로 "디바이스 런타임에 몇 번 찍히는가"를 보여주는 게 핵심이라 그냥 두면 `[device]`
줄이 전부 빠집니다. 노트북1의 `show_device_printf()` 헬퍼를 그대로 가져와 씁니다.

In [ ]:
import os
import sys
import ctypes
import tempfile
import contextlib


@contextlib.contextmanager
def show_device_printf():
    """cute.printf가 fd 1에 직접 쓰는 내용을 캡처해서 with 블록이 끝난 뒤 print합니다."""
    sys.stdout.flush()
    saved_fd1 = os.dup(1)
    tmp = tempfile.NamedTemporaryFile(delete=False)
    tmp_path = tmp.name
    tmp.close()
    tmp_fd = os.open(tmp_path, os.O_WRONLY)
    os.dup2(tmp_fd, 1)
    try:
        yield
        cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
        ctypes.CDLL(None).fflush(None)
    finally:
        sys.stdout.flush()
        os.dup2(saved_fd1, 1)
        os.close(saved_fd1)
        os.close(tmp_fd)
    with open(tmp_path, "r") as f:
        captured = f.read()
    os.unlink(tmp_path)
    if captured:
        print(captured, end="")

## 1. 루프 두 종류

같은 일 — 인덱스 `0 .. N-1`을 훑는 것 — 을 두 방식으로 씁니다. 차이는 루프 드라이버 하나뿐이고,
그 한 단어가 **어디서 반복이 일어나는지**를 결정합니다.

| 루프 | 정체 | 어디서 반복 | bound | 본문 트레이스 횟수 | 생성물 |
|---|---|---|---|---|---|
| `cutlass.range_constexpr(N)` | **메타프로그램**(파이썬) | **호스트, 트레이스타임** | 반드시 파이썬 `int` | **`N`번** | `N`벌 일자 코드(완전 언롤) |
| `cutlass.range(N)` | **스테이지드**(GPU) | **디바이스, 런타임** | 파이썬 `int` *또는* 동적 `Int32` | **한 번** | `scf.for` 하나, 하드웨어가 반복 |

이 경계를 각 본문의 트레이스타임 `print`로 증명합니다. `print`는 그냥 파이썬이라, 호스트가
본문을 걷는 만큼만 찍힙니다 — 메타 루프는 **`N`줄**, 스테이지드 루프는 **한 줄**. 메타
루프에서는 인덱스 `i`가 진짜 파이썬 `int`라서(그래서 `i`가 호스트 쪽에서 누적된 총합이
커널에 상수로 *박힙니다*), 스테이지드 루프에서는 `i`가 GPU가 매 트립마다 공급하는 스테이지드
`Int32`라서 일이 디바이스에서 벌어집니다.

In [ ]:
@cute.kernel
def loop_unrolled(N: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = 0  # a plain Python int -- accumulated at TRACE time
        # N is a Constexpr (Python int), so this body runs N times right here, in Python.
        # The print fires once per iteration; `total` is computed on the host and baked in.
        for i in cutlass.range_constexpr(N):
            print(f"[trace] range_constexpr body, i = {i}  (a Python int)")
            total += i
        # `total` is now a constant -- cute.printf just emits that baked number.
        cute.printf("[device] unrolled: sum of 0..{} baked at trace = {}", N - 1, total)


@cute.jit
def loop_unrolled_host(N: cutlass.Constexpr):
    loop_unrolled(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

스테이지드 버전은 딱 두 단어만 다릅니다 — `N`이 동적 `cutlass.Int32`(bound가 커널을 쓸 때가
아니라 *launch* 시점에 정해짐)이고 루프가 `cutlass.range`라는 것. 이제 본문은 **한 번만**
트레이스되고, 누산기는 GPU 루프를 통해 알아서 이어지고, 합은 **디바이스**에서 계산됩니다.

In [ ]:
@cute.kernel
def loop_staged(N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = cutlass.Int32(0)
        # N is a dynamic Int32, so this is a real GPU loop: the body traces ONCE and the
        # print fires ONCE. `i` is a staged Int32 -- the GPU supplies its value each trip.
        for i in cutlass.range(N):
            print(f"[trace] range body, i is a staged {type(i).__name__} (traced once)")
            total = total + i
        cute.printf("[device] staged: sum of 0..{} computed on the GPU = {}", N - 1, total)


@cute.jit
def loop_staged_host(N: cutlass.Int32):
    loop_staged(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

## 2. 둘 다 돌려서 트레이스를 지켜보기

배열도, 데이터 준비도 없습니다 — 커널 두 개를 스테이징하고 print만 읽으면 됩니다. **트레이스**
print는 각 커널이 *컴파일*되는 동안(호스트, 파이썬) 찍힙니다 — 메타 루프는 `N`줄, 스테이지드
루프는 한 줄. **디바이스** print는 launch 이후, sync하고 나서 찍히고, 둘 다 같은 합을
보고합니다.

In [ ]:
N = 4

with show_device_printf():
    print("--- staging loop_unrolled (range_constexpr) ---")
    loop_unrolled_host(N)            # N trace lines fire HERE, during compile

    print("\n--- staging loop_staged (range) ---")
    loop_staged_host(N)              # exactly ONE trace line fires here

# Expected output (device 줄은 show_device_printf() 덕분에 여기서 보입니다):
# --- staging loop_unrolled (range_constexpr) ---
# [trace] range_constexpr body, i = 0  (a Python int)
# [trace] range_constexpr body, i = 1  (a Python int)
# [trace] range_constexpr body, i = 2  (a Python int)
# [trace] range_constexpr body, i = 3  (a Python int)
#
# --- staging loop_staged (range) ---
# [trace] range body, i is a staged Int32 (traced once)   <- exactly ONE line
#
# [device] unrolled: sum of 0..3 baked at trace = 6
# [device] staged: sum of 0..3 computed on the GPU = 6

## 3. 메타 루프는 그냥 파이썬이라 — `raise`도 됩니다

`range_constexpr`은 트레이스타임에 파이썬 인터프리터에서 도니까, 본문은 파이썬이 할 수 있는
건 뭐든 합니다 — 계산, 파이썬 값으로 분기, 헬퍼 호출, 심지어 **예외 raise**까지. 여기서
raise하면 *컴파일*이 중단됩니다 — GPU까지 가지도 않습니다. 이게 메타프로그램의 핵심입니다 —
디바이스 명령이 하나도 존재하기 전에, 그냥 파이썬으로 커널을 검증하고 다듬을 수 있습니다.
(이걸 `range` 본문 안에서 해보면 테스트할 파이썬 `i`가 없습니다 — 그 검사 자체가 진짜 GPU
코드가 돼야 합니다.)

In [ ]:
@cute.kernel
def checked_unroll(N: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    
    for i in cutlass.range_constexpr(N):
        # This runs in Python, WHILE TRACING. Ordinary Python control flow works --
        # including raising, which stops compilation before any IR is emitted for i>=3.
        if cutlass.const_expr(i == 3):
            raise ValueError(
                f"meta-loop guard tripped at i={i}: this is a Python exception raised "
                "during TRACING, before any GPU instruction is emitted"
            )
        cute.printf("[device] checked i={}", i)


@cute.jit
def checked_unroll_host(N: cutlass.Constexpr):
    checked_unroll(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

In [ ]:
print("N=3: the meta loop never reaches i==3, so it traces + runs cleanly:")
with show_device_printf():
    checked_unroll_host(3)

print("\nN=8: the meta loop hits i==3 and raises -- DURING tracing, on the host:")
try:
    checked_unroll_host(8)   # raises while the Python body is being walked
except ValueError as e:
    print(f"  caught at trace time: {e}")

# Expected output (device 줄은 show_device_printf() 덕분에 여기서 보입니다):
# N=3: ... traces + runs cleanly:
# [device] checked i=0
# [device] checked i=1
# [device] checked i=2
#
# N=8: ... raises -- DURING tracing, on the host:
#   caught at trace time: meta-loop guard tripped at i=3: this is a Python exception ...

## 4. 부분 언롤: `range(N, unroll=K)`

스테이지드 루프도 언롤할 수 있습니다 — 다만 끝까지는 아닙니다. `cutlass.range(N, unroll=K)`는
진짜 GPU 루프를 유지하면서(bound는 동적으로 남고, 본문은 여전히 **한 번만** 트레이스)
컴파일러에게 트립당 본문 `K`벌을 내라고 요청합니다 — 그래서 루프는 `N/K`번 돌고 본문은 `K`
폭입니다. 딱 중간 지점입니다 — 기본값보다 루프 제어 명령은 적고 명령어 수준 병렬성은 높지만,
완전 언롤의 코드 크기 폭증은 없습니다. 결과는 동일 — 생성되는 루프 모양만 바뀝니다.
(`unroll=0`/`1`은 그냥 스테이지드 루프, `unroll_full=True`는 동적 루프를 완전히 평평하게
폅니다.)

In [ ]:
@cute.kernel
def loop_unroll4(N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = cutlass.Int32(0)
        # Still a GPU loop (N is dynamic) -- the body traces once; the COMPILER emits 4
        # bodies per trip. So the trace print still fires exactly once.
        for i in cutlass.range(N, unroll=4):
            print(f"[trace] unroll=4 body, traced once (i is staged {type(i).__name__})")
            total = total + i
        cute.printf("[device] unroll=4: sum of 0..{} = {}", N - 1, total)


@cute.jit
def loop_unroll4_host(N: cutlass.Int32):
    loop_unroll4(N).launch(grid=(1, 1, 1), block=(1, 1, 1))


with show_device_printf():
    loop_unroll4_host(N)

# Expected output (device 줄은 show_device_printf() 덕분에 여기서 보입니다):
# [trace] unroll=4 body, traced once (i is staged Int32)   <- still ONE trace line
# [device] unroll=4: sum of 0..3 = 6

## 5. 분기 두 종류

분기도 루프와 똑같이 갈립니다:

- `if cutlass.const_expr(cond):` — `cond`가 트레이스타임에 고정된 파이썬 값이라, 호스트가
  **한쪽**만 골라 그것만 컴파일하고, 나머지는 트레이스조차 안 됩니다. 각 쪽의 트레이스타임
  `print`가 이걸 증명합니다 — 선택된 쪽만 찍힙니다.
- 평범한 `if <스테이지드값>:` — 조건이 스레드별 디바이스 값이라 **양쪽 다** 컴파일되고 GPU가
  스레드마다 고릅니다(진짜 `scf.if`). 그냥 파이썬 `if`를 쓰는 건 똑같지만, 조건이 동적이라
  DSL이 알아서 스테이지합니다.

여기도 배열은 없습니다 — `Constexpr` 플래그 하나와 스레드별 `tidx`로 분기합니다.

In [ ]:
@cute.kernel
def branch_demo(USE_DOUBLE: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    # COMPILE-TIME branch: USE_DOUBLE is a Python bool, so exactly ONE side is traced and
    # emitted -- the other side's print never fires. `factor` is baked from the taken side.
    if cutlass.const_expr(USE_DOUBLE):
        print("[trace] const_expr branch: doubling side compiled in")
        factor = 2
    else:
        print("[trace] const_expr branch: identity side compiled in")
        factor = 1
    # RUNTIME branch: tx is a staged value, so BOTH sides compile and the GPU picks per
    # thread. Plain Python if -- the DSL stages it into a real scf.if.
    if tx == 0:
        cute.printf("[device] thread {} -> first lane (baked factor = {})", tx, factor)
    else:
        cute.printf("[device] thread {} -> other lane (baked factor = {})", tx, factor)


@cute.jit
def branch_demo_host(USE_DOUBLE: cutlass.Constexpr):
    branch_demo(USE_DOUBLE).launch(grid=(1, 1, 1), block=(2, 1, 1))

In [ ]:
for use_double in (True, False):
    with show_device_printf():
        print(f"--- staging branch_demo(USE_DOUBLE={use_double}) ---")
        branch_demo_host(use_double)        # only the TAKEN const_expr side prints at trace
    print()

# Expected output (device 줄은 show_device_printf() 덕분에 여기서 보입니다,
# only ONE const_expr print per staging -- the taken side):
# --- staging branch_demo(USE_DOUBLE=True) ---
# [trace] const_expr branch: doubling side compiled in
# [device] thread 0 -> first lane (baked factor = 2)
# [device] thread 1 -> other lane (baked factor = 2)
#
# --- staging branch_demo(USE_DOUBLE=False) ---
# [trace] const_expr branch: identity side compiled in
# [device] thread 0 -> first lane (baked factor = 1)
# [device] thread 1 -> other lane (baked factor = 1)

### 실측 검증 (2026-09-27, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(스크립트가 아니라 실제 셀 순서대로 하나의 커널에서) Colab
L4에서 실행해 확인했습니다. 13개 셀 전부 에러 없이 통과했고, 모든 실행 셀의 출력이 원문
"Expected output"과 내용·순서가 일치했습니다(단, 원문 주석의 빈 줄 하나는 실제 출력엔 없음
— 사소한 서식 차이).

먼저 원문 그대로(`show_device_printf()` 없이) 실행해서 실제로 `[device]` 줄이 전부 빠지는지
확인했습니다 — 노트북1보다 이 챕터에서 영향이 더 큽니다: 섹션 2/3/4/5 네 곳 모두 device
printf에 의존하는데, 그중 섹션 5(두 종류 분기)는 device 줄이 콘텐츠의 절반을 차지합니다.
`show_device_printf()`를 넣은 뒤로는 이 네 곳 모두 원문 그대로 재현됩니다.

커널 정의 자체는 원본 그대로입니다. 실행 셀은 `with show_device_printf():`로 감쌌고,
원래 각 실행 셀에 있던 `cutlass.cuda.stream_sync(...)` 호출은 이 헬퍼 안으로 옮겼습니다
(헬퍼가 `yield` 뒤에 자체적으로 `stream_sync`+`fflush`를 호출).

## 직접 해보기

1. **트레이스 print 세어보기.** `N`을 8로 올리고 섹션 2를 다시 실행해 보세요 — 메타 루프는
   이제 8줄을 찍고, 스테이지드 루프는 여전히 정확히 한 줄입니다.
2. **constexpr bound를 깨보기.** `loop_unrolled`의 `N`을 `cutlass.Constexpr`에서
   `cutlass.Int32`로 바꾸고 런타임 값을 넘겨 보세요 — `range_constexpr`이 raise합니다, bound가
   트레이스타임에 알려진 파이썬 `int`여야 하니까요. 그 루프를 `cutlass.range(N)`으로 바꾸면
   해결됩니다.
3. **가드를 스테이지드 루프 안으로 옮겨보기.** 섹션 3의 `if i == 3: raise ...`를
   `cutlass.range(N)` 루프 안에 넣어 보세요. 트립마다 raise되지 *않습니다* — `i`가 스테이지드
   값이라 `i == 3`이 디바이스 비교가 되지, 파이썬 비교가 아니니까요. 메타 전용 트릭이 사라진
   겁니다.
4. **const_expr 분기를 런타임으로 만들어보기.** 섹션 5에서 `cutlass.const_expr(...)`를
   빼고 `USE_DOUBLE`을 `cutlass.Int32` 플래그로 넘겨 `> 0`으로 비교해 보세요 — 이제 *양쪽* 다
   컴파일되고(트레이스 print도 둘 다 찍힘) GPU가 런타임에 고릅니다, `tx == 0` 가드가 이미
   하던 것과 똑같이.

다음 `03_diagnostics`는 컴파일러 진단(`warnings{...}`/`remarks{...}` 옵션)을 다루고,
`04_zero_cost_abstraction`은 클래스·다형성이 트레이스타임 파이썬이라 컴파일되며 사라진다는 걸
PTX로 증명합니다. (두 노트북 모두 이 저장소에 있습니다: `03_diagnostics.ipynb`, `04_zero_cost_abstraction.ipynb`.)